## CMT307 Lab 1, Exercise 2

**This exercise prepares you for using the Colab, data reading and your first data understanding practice.**

The data set is from UCI Machine Learning Repository https://archive.ics.uci.edu/ml/datasets/Online+Shoppers+Purchasing+Intention+Dataset#, visit the web page for information about the data.

## Get Data

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('online_shoppers_intention.csv')

In [2]:
df
print('The shape of the dataset is', df.shape)

The shape of the dataset is (12330, 18)


In [3]:
pd.set_option('display.max_columns', None, 'display.expand_frame_repr', False)

# Have a first look at the dataset
print(df.head(5))

   Administrative  Administrative_Duration  Informational  Informational_Duration  ProductRelated  ProductRelated_Duration  BounceRates  ExitRates  PageValues  SpecialDay Month  OperatingSystems  Browser  Region  TrafficType        VisitorType  Weekend  Revenue
0               0                      0.0              0                     0.0               1                 0.000000         0.20       0.20         0.0         0.0   Feb                 1        1       1            1  Returning_Visitor    False    False
1               0                      0.0              0                     0.0               2                64.000000         0.00       0.10         0.0         0.0   Feb                 2        2       1            2  Returning_Visitor    False    False
2               0                      0.0              0                     0.0               1                 0.000000         0.20       0.20         0.0         0.0   Feb                 4        1       9   

## Missing Values?
Check if the dataset contains missing values

In [4]:
# Check for missing values
df.isna().sum()

Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64

No missing data which is very helpful!

## Split into input and output pairs
Split the dataset into input (denoted by x) and output (denoted by y) pairs. The output is the target that the machine learning model will be built to predict for.

In [ ]:
print(df['Revenue'].value_counts())

In [ ]:
# Convert boolean into binary
df['Revenue'] = [1 if  x == True else 0 for x in df['Revenue']]

In [ ]:
# Matches the above value counts
print(df['Revenue'].value_counts())

In [ ]:
# Split into x, input, and y, output
x = df.drop('Revenue', 1)
y = df.Revenue
y

In [ ]:
# Quick look at x
print(x.head(5))

## Data Exploration

In [ ]:
# Have a look at the feature types and size of dataset
df.info()

In [ ]:
# Look into the three different webpages that shoppers can visit
print(df[['Administrative', 'Informational', 'ProductRelated']].describe())
df[['Administrative', 'Informational', 'ProductRelated']].boxplot()

The boxplot for ProductRelated is very stretched, shwoing a wide variation in the number of Product Related pages that were visited. Some people were clearly very indecisive about what to purchase.

In [ ]:
# https://github.com/aprilypchen/depy2016/blob/master/DePy_Talk.ipynb
# This is a function that finds the outliers that are outside of 1.5xIQR. 
# It returns what value the outlier has, and where in the dataset they are
def find_outliers_tukey(x):
    q1 = np.percentile(x, 25)
    q3 = np.percentile(x, 75)
    iqr = q3-q1 
    floor = q1 - 1.5*iqr
    ceiling = q3 + 1.5*iqr
    outlier_indices = list(x.index[(x < floor)|(x > ceiling)])
    outlier_values = list(x[outlier_indices])

    return outlier_indices, outlier_values

In [ ]:
outlier_indices, outlier_values = find_outliers_tukey(df['Administrative'])
print(len(np.sort(outlier_values)))

In [ ]:
outlier_indices, outlier_values = find_outliers_tukey(df['Informational'])
print(len(np.sort(outlier_values)))

In [ ]:
outlier_indices, outlier_values = find_outliers_tukey(df['ProductRelated'])
print(len(np.sort(outlier_values)))

Surprisingly, there are more outliers for Informational, than ProductRelated. But that is because the upper quartile for Informational is 0, so basically any Informational value not equal to zero is an outlier. Because of the great numbers of Tukey outliers, this exwecise is not going to remove them as they that would remove too much of the data, thus lowering the reliability of the model. 

In [ ]:
df.boxplot()

The massive boxplot is ProductRelated_Duration, where people are seen to be browsing for up to almost 17 hours. Again, this doesn't seem right but this will have a negligable effect on the model.

In [ ]:
# Boxplot of durations spent on different webpages.
print(df[['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration']].describe())
df[['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration']].boxplot()

In [ ]:
# https://github.com/aprilypchen/depy2016/blob/master/DePy_Talk.ipynb
# These histograms layer the outcomes of when revenue was and wasn't made. 
# Exit rate is the number of exits/number of pageviews for a particular page
import matplotlib.pyplot as plt
def plot_histogram_dv(x,y):
    plt.hist(list(x[y==0]), alpha=0.5, label='Outcome=0', color = 'blue')
    plt.hist(list(x[y==1]), alpha=0.5, label='Outcome=1', color = 'yellow')
    plt.title("Histogram of '{var_name}' by Outcome Category".format(var_name=x.name))
    plt.xlabel("Value")
    plt.ylabel("Frequency")
    plt.legend(loc='upper right')
    plt.show()
plot_histogram_dv(df['ExitRates'], y)
df['ExitRates'].describe()

In [ ]:
# Bounce rate is the total number of one page visits/ total number of visits
df['BounceRates'].describe()
plot_histogram_dv(df['BounceRates'], y)


In [ ]:
df['ProductRelated_Duration'].describe()
plot_histogram_dv(df['ProductRelated_Duration'], y)


In [ ]:
plot_histogram_dv(df['Administrative_Duration'], y)
(df['Administrative_Duration']).describe()

In [ ]:
plot_histogram_dv(df['Informational_Duration'], y)
df['Informational_Duration'].describe()

In [ ]:
plot_histogram_dv(df['SpecialDay'], y)

In [ ]:
plot_histogram_dv(df['PageValues'], y)
df['PageValues'].describe()

Interestingly, as the average page value increases, the likelihood of revenue being made actually outweighs the likelihood of revenue not being made. This may be explained by the page value increasing as a customer maybe reaches the payment page on the website, where they are most likely to continue and end up purchasing an item. 

In [ ]:
plot_histogram_dv(df['TrafficType'], y)

In [ ]:
plot_histogram_dv(df['OperatingSystems'], y)
# This is not exponential due to it being a categorical variable.

In [ ]:
plot_histogram_dv(df['Region'], y)

In [ ]:
plot_histogram_dv(df['Browser'], y)

In [ ]:
# https://medium.com/@szabo.bibor/how-to-create-a-seaborn-correlation-heatmap-in-python-834c0686b88e
# The heatmap below gives a correlation of each feature against every other feature. 
# For categorical varibales it doesn't make much sense though.
import seaborn as sns
sns.heatmap(df.corr());
sns.set(font_scale=2)
heatmap = sns.heatmap(df.corr(), vmin=-1, vmax=1, cbar = False)
heatmap.set_title('Correlation Heatmap', fontdict={'fontsize':12}, pad=12);

In [ ]:
df.plot(kind='scatter', x='BounceRates', y='ExitRates', color = 'blue')

In [ ]:
df.plot(kind='scatter', x='Informational_Duration', y='ProductRelated_Duration', color = 'blue')

In [ ]:
# Barplot showing data imbalance of Revenue feature
df['Revenue'].value_counts().plot(kind='bar', title= 'Barplot of Imbalance of Revenue')

In [ ]:
df['Browser'].value_counts().plot(kind='bar', title= 'Barplot of Imbalance of Browser')

In [ ]:
# Obviously weekend only makes up 2 out of 7 days a week
df['Weekend'].value_counts().plot(kind='bar', title= 'Barplot of Imbalance of Weekend')

In [ ]:
df['OperatingSystems'].value_counts().plot(kind='bar', title= 'Barplot of Imbalance of OperatingSystems')

As shown by the barcharts above, the data is pretty imbalanced. Thus, data balancing method may need to be considered to balance the data.

## Exercise

You can now practice the above steps by usign the Abalone Data Set available from https://archive.ics.uci.edu/ml/datasets/Abalone